<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/JARVIS_OpenMM_CoarseGraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://cg-openmm.readthedocs.io/en/latest/examples/replica_exchange_simulation.html

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda --version

In [ ]:
import condacolab
condacolab.check()

In [ ]:
!pip install -q -U git+https://github.com/shirtsgroup/cg_openmm

In [ ]:
!pip install -q -U git+https://github.com/shirtsgroup/analyze_foldamers.git

In [ ]:
!conda install mdtraj mpi4py numpy openmm openmmtools physical_validation -y --quiet

In [ ]:
!wget https://raw.githubusercontent.com/shirtsgroup/cg_openmm/773a0c76ddd6d4192bfdd7c66affbcec86d5530b/examples/run_standard_MD_simulation/initial_structure_trial_1.pdb -O initial_structure_trial_1.pdb

In [ ]:
import os
import pickle
import time

import cg_openmm
import numpy as np
from cg_openmm.cg_model.cgmodel import CGModel
from cg_openmm.simulation.tools import run_simulation
from openmm import unit
from openmm.app.pdbfile import PDBFile

# This example demonstrates how to run a standard (single temperature) NVT simulation
# for a coarse-grained model in OpenMM.

sim_begin = time.perf_counter()

#---------------------------#
# Simulation job parameters #
#---------------------------#

# Set output directory:
output_directory = "output"
if not os.path.exists(output_directory):
    os.mkdir(output_directory)

# Replica exchange simulation settings:
total_simulation_time = 5.0 * unit.nanosecond
simulation_time_step = 5.0 * unit.femtosecond
total_steps = int(np.floor(total_simulation_time / simulation_time_step))
temperature = 300.0 * unit.kelvin
collision_frequency = 5 / unit.picosecond
print_frequency = 5000  # Number of steps to skip when printing output

include_bond_forces = True
include_bond_angle_forces = True
include_nonbonded_forces = True
include_torsion_forces = True
constrain_bonds = False

#--------------------------------------------#
# Particle definitions and oligomer topology #
#--------------------------------------------#

mass = 100.0 * unit.amu

# Mass and charge are defaults.
# Backbone particle:
bb = {
    "particle_type_name": "bb",
    "sigma": 2.25 * unit.angstrom,
    "epsilon": 1.50 * unit.kilojoules_per_mole,
    "mass": mass
}

# Sidechain particle:
sc = {
    "particle_type_name": "sc",
    "sigma": 3.50 * unit.angstrom,
    "epsilon": 5.00 * unit.kilojoules_per_mole,
    "mass": mass
}

# Monomer definition:
A = {
    "monomer_name": "A",
    "particle_sequence": [bb, sc],
    "bond_list": [[0, 1]],
    "start": 0,
    "end": 0,
}

# Residue sequence:
sequence = 24 * [A]

#--------------------------#
# Harmonic bond parameters #
#--------------------------#

# Bond definitions:
bond_lengths = {"default_bond_length": 2.44 * unit.angstrom}

bond_force_constants = {
    "default_bond_force_constant": 15000 * unit.kilojoule_per_mole / unit.nanometer / unit.nanometer
}

#---------------------------#
# Harmonic angle parameters #
#---------------------------#

# Bond angle definitions:
bond_angle_force_constants = {
    "default_bond_angle_force_constant": 150 * unit.kilojoule_per_mole / unit.radian / unit.radian
}

equil_bond_angles = {
    "default_equil_bond_angle": 127.5 * unit.degrees,
    "bb_bb_bb_equil_bond_angle": 105.5 * unit.degrees}

#-----------------------------#
# Periodic torsion parameters #
#-----------------------------#

# Torsion angle definitions:
torsion_force_constants = {
    "default_torsion_force_constant": 0.0 * unit.kilojoule_per_mole,
    "bb_bb_bb_bb_torsion_force_constant": 5 * unit.kilojoule_per_mole}

torsion_phase_angles = {
    "sc_bb_bb_sc_torsion_phase_angle": 0 * unit.degrees,
    "bb_bb_bb_bb_torsion_phase_angle": (16.7-180) * unit.degrees,
    "bb_bb_bb_sc_torsion_phase_angle": 0 * unit.degrees,
}

torsion_periodicities = {
    "default_torsion_periodicity": 1,
}

# Get initial positions from file:
pdb_path = "initial_structure_trial_1.pdb"
positions = PDBFile(pdb_path).getPositions()

# Build a coarse grained model:
cgmodel = CGModel(
    particle_type_list=[bb, sc],
    bond_lengths=bond_lengths,
    bond_force_constants=bond_force_constants,
    bond_angle_force_constants=bond_angle_force_constants,
    torsion_force_constants=torsion_force_constants,
    equil_bond_angles=equil_bond_angles,
    torsion_phase_angles=torsion_phase_angles,
    torsion_periodicities=torsion_periodicities,
    include_nonbonded_forces=include_nonbonded_forces,
    include_bond_forces=include_bond_forces,
    include_bond_angle_forces=include_bond_angle_forces,
    include_torsion_forces=include_torsion_forces,
    constrain_bonds=constrain_bonds,
    positions=positions,
    sequence=sequence,
    monomer_types=[A],
)

# Store the cg model so that we can do various analyses.
cgmodel.export("stored_cgmodel.pkl")

# Run the simulation
run_simulation(
    cgmodel,
    total_simulation_time,
    simulation_time_step,
    temperature,
    friction=collision_frequency,
    print_frequency=print_frequency,
    output_directory=output_directory,
)


sim_end = time.perf_counter()

print(f'simulation run time: {sim_end-sim_begin}')

In [ ]:
#!~/anaconda3/bin/python

import os
import time

import numpy as np
import openmm
from cg_openmm.cg_model.cgmodel import CGModel
from cg_openmm.parameters.reweight import get_temperature_list
from cg_openmm.simulation.rep_exch import *
from openmm import unit
from openmmtools.cache import global_context_cache

# This example demonstrates how to run a OpenMM replica exchange simulation
# using a CGModel object.


rep_exch_begin = time.perf_counter()

#---------------------------------#
# Replica exchange job parameters #
#---------------------------------#

# Set output directory:
output_directory = "output"
if not os.path.exists(output_directory):
    os.mkdir(output_directory)
overwrite_files = True  # overwrite files.

# Use CUDA platform for GPU acceleration:
global_context_cache.platform = openmm.Platform.getPlatformByName("CUDA")

# Replica exchange simulation settings:
total_simulation_time = 50.0 * unit.nanosecond
simulation_time_step = 5.0 * unit.femtosecond
total_steps = int(np.floor(total_simulation_time / simulation_time_step))
output_data = os.path.join(output_directory, "output.nc")

number_replicas = 12
min_temp = 200.0 * unit.kelvin
max_temp = 600.0 * unit.kelvin
# Use logarithmic temperature spacing:
temperature_list = get_temperature_list(min_temp, max_temp, number_replicas)

exchange_frequency = 200 # Number of steps between exchange attempts
collision_frequency = 5 / unit.picosecond

include_bond_forces = True
include_bond_angle_forces = True
include_nonbonded_forces = True
include_torsion_forces = True
constrain_bonds = False

#--------------------------------------------#
# Particle definitions and oligomer topology #
#--------------------------------------------#

mass = 100.0 * unit.amu

# Mass and charge are defaults.
# Backbone particle:
bb = {
    "particle_type_name": "bb",
    "sigma": 2.25 * unit.angstrom,
    "epsilon": 1.50 * unit.kilojoules_per_mole,
    "mass": mass
}

# Sidechain particle:
sc = {
    "particle_type_name": "sc",
    "sigma": 3.50 * unit.angstrom,
    "epsilon": 5.00 * unit.kilojoules_per_mole,
    "mass": mass
}

# Monomer definition:
A = {
    "monomer_name": "A",
    "particle_sequence": [bb, sc],
    "bond_list": [[0, 1]],
    "start": 0,
    "end": 0,
}

# Residue sequence:
sequence = 24 * [A]

#--------------------------#
# Harmonic bond parameters #
#--------------------------#

# Bond definitions:
bond_lengths = {"default_bond_length": 2.44 * unit.angstrom}

bond_force_constants = {
    "default_bond_force_constant": 15000 * unit.kilojoule_per_mole / unit.nanometer / unit.nanometer
}

#---------------------------#
# Harmonic angle parameters #
#---------------------------#

# Bond angle definitions:
bond_angle_force_constants = {
    "default_bond_angle_force_constant": 150 * unit.kilojoule_per_mole / unit.radian / unit.radian
}

equil_bond_angles = {
    "default_equil_bond_angle": 127.5 * unit.degrees,
    "bb_bb_bb_equil_bond_angle": 105.5 * unit.degrees}

#-----------------------------#
# Periodic torsion parameters #
#-----------------------------#

# Torsion angle definitions:
torsion_force_constants = {
    "default_torsion_force_constant": 0.0 * unit.kilojoule_per_mole,
    "bb_bb_bb_bb_torsion_force_constant": 5 * unit.kilojoule_per_mole}

torsion_phase_angles = {
    "sc_bb_bb_sc_torsion_phase_angle": 0 * unit.degrees,
    "bb_bb_bb_bb_torsion_phase_angle": (16.7-180) * unit.degrees,
    "bb_bb_bb_sc_torsion_phase_angle": 0 * unit.degrees,
}

torsion_periodicities = {
    "default_torsion_periodicity": 1,
}

# Get initial positions from file:
pdb_path = "initial_structure_trial_1.pdb"
positions = PDBFile(pdb_path).getPositions()

# Build a coarse grained model:
cgmodel = CGModel(
    particle_type_list=[bb, sc],
    bond_lengths=bond_lengths,
    bond_force_constants=bond_force_constants,
    bond_angle_force_constants=bond_angle_force_constants,
    torsion_force_constants=torsion_force_constants,
    equil_bond_angles=equil_bond_angles,
    torsion_phase_angles=torsion_phase_angles,
    torsion_periodicities=torsion_periodicities,
    include_nonbonded_forces=include_nonbonded_forces,
    include_bond_forces=include_bond_forces,
    include_bond_angle_forces=include_bond_angle_forces,
    include_torsion_forces=include_torsion_forces,
    constrain_bonds=constrain_bonds,
    positions=positions,
    sequence=sequence,
    monomer_types=[A],
)

# Store the cg model so that we can do various analyses.
cgmodel.export("stored_cgmodel.pkl")

if not os.path.exists(output_data) or overwrite_files == True:
    run_replica_exchange(
        cgmodel.topology,
        cgmodel.system,
        cgmodel.positions,
        friction=collision_frequency,
        temperature_list=temperature_list,
        simulation_time_step=simulation_time_step,
        total_simulation_time=total_simulation_time,
        exchange_frequency=exchange_frequency,
        output_data=output_data,
    )
else:
    print("Replica output files exist")

rep_exch_end = time.perf_counter()

print(f'replica exchange run time: {rep_exch_end-rep_exch_begin}')

In [ ]:


import os
import pickle

from cg_openmm.thermo.calc import *
from openmm import unit

# This example demonstrates how to calculate heat capacity as a function of temperature from
# replica exchange energies, with uncertainties estimated using pyMBAR.

# Note: process_replica_exchange_data should first be run to determine the determine the start
# of the production region and energy decorrelation time.


# Job settings
output_directory = 'output'
output_data = os.path.join(output_directory, "output.nc")

# Load in trajectory stats:
analysis_stats = pickle.load(open("analysis_stats_discard_20ns.pkl","rb"))

# Read the simulation coordinates for individual temperature replicas
C_v, dC_v, new_temperature_list, FWHM, Tm, Cv_height, N_eff = get_heat_capacity(
    output_data=output_data,
    frame_begin=analysis_stats["production_start"],
    sample_spacing=analysis_stats["energy_decorrelation"],
    num_intermediate_states=3,
    plot_file="heat_capacity.pdf",
)

print(f"T({new_temperature_list[0].unit})  Cv({C_v[0].unit})  dCv({dC_v[0].unit})")
for i, C in enumerate(C_v):
    print(f"{new_temperature_list[i]._value:>8.2f}{C_v[i]._value:>10.4f} {dC_v[i]._value:>10.4f}")

In [ ]:
from simtk import unit
from cg_openmm.cg_model.cgmodel import CGModel

# Specify backbone (bb) and sidechain (sc) particle parameters:
sigma = 0.3 * unit.nanometer
epsilon = 2 * unit.kilojoule_per_mole
mass = 100 * unit.amu

bb = {"particle_type_name": "bb", "sigma": sigma, "epsilon": epsilon, "mass": mass}
sc = {"particle_type_name": "sc", "sigma": sigma, "epsilon": epsilon, "mass": mass}

# Define monomer (residue):
A = {
    "monomer_name": "A",
    "particle_sequence": [bb, sc],
    "bond_list": [[0, 1]],
    "start": 0,
    "end": 0}

# Specify bonded parameters:
bond_lengths = {
    "default_bond_length": 0.35 * unit.nanometer,
    "bb_bb_bb_bond_length": 0.40 * unit.nanometer}

bond_force_constants = {
    "default_bond_force_constant": 1000 * unit.kilojoule_per_mole / unit.nanometer**2}

equil_bond_angles = {
    "default_equil_bond_angle": 120.0 * unit.degrees,
    "bb_bb_bb_equil_bond_angle": 150.0 * unit.degrees}

bond_angle_force_constants = {
    "default_bond_angle_force_constant": 100.0 * unit.kilojoule_per_mole / unit.radian**2}

torsion_phase_angles = {
    "default_torsion_phase_angle": 150 * unit.degrees}

torsion_force_constants = {
    "default_torsion_force_constant": 2.0 * unit.kilojoule_per_mole,
    "bb_bb_bb_bb_torsion_force_constant": 5.0 * unit.kilojoule_per_mole}

torsion_periodicities = {
    "default_torsion_periodicity": 1}

# Define oligomer sequence:
sequence = 12 * [A]

# Initial particle positions determined from random builder

cgmodel = CGModel(
    particle_type_list=[bb, sc],
    bond_lengths=bond_lengths,
    bond_force_constants=bond_force_constants,
    bond_angle_force_constants=bond_angle_force_constants,
    torsion_force_constants=torsion_force_constants,
    equil_bond_angles=equil_bond_angles,
    torsion_phase_angles=torsion_phase_angles,
    torsion_periodicities=torsion_periodicities,
    include_nonbonded_forces=True,
    include_bond_forces=True,
    include_bond_angle_forces=True,
    include_torsion_forces=True,
    constrain_bonds=False,
    sequence=sequence,
    monomer_types=[A],
)

In [ ]:
cgmodel.bond_list

In [ ]:
cgmodel.topology.createTopology()

In [ ]:
cgmodel.export("stored_cgmodel.pkl")

In [ ]:
import os

import numpy as np
import simtk.openmm as openmm
from cg_openmm.cg_model.cgmodel import CGModel
from cg_openmm.parameters.reweight import get_temperature_list
from cg_openmm.simulation.rep_exch import *
from openmmtools.cache import global_context_cache
from simtk import unit
import pickle

# Set output directory:
output_directory = "output"
if not os.path.exists(output_directory):
    os.mkdir(output_directory)

# Use CUDA platform for GPU acceleration:
global_context_cache.platform = openmm.Platform.getPlatformByName("CUDA")

# Replica exchange simulation settings:
total_simulation_time = 50.0 * unit.nanosecond
simulation_time_step = 5.0 * unit.femtosecond
total_steps = int(np.floor(total_simulation_time / simulation_time_step))
output_data = os.path.join(output_directory, "output.nc")

number_replicas = 12
min_temp = 200.0 * unit.kelvin
max_temp = 600.0 * unit.kelvin
# Use logarithmic temperature spacing:
temperature_list = get_temperature_list(min_temp, max_temp, number_replicas)

exchange_frequency = 200 # Number of steps between exchange attempts
collision_frequency = 5 / unit.picosecond

# Load in a cgmodel:
cgmodel = pickle.load(open("stored_cgmodel.pkl","rb"))

run_replica_exchange(
    cgmodel.topology,
    cgmodel.system,
    cgmodel.positions,
    friction=collision_frequency,
    temperature_list=temperature_list,
    simulation_time_step=simulation_time_step,
    total_simulation_time=total_simulation_time,
    exchange_frequency=exchange_frequency,
    output_data=output_data,
)

In [ ]:
cgmodel.

In [ ]:
!git clone https://github.com/shirtsgroup/cg_openmm.git

In [ ]:
import os
os.chdir("cg_openmm")

In [ ]:
!python setup.py install

In [ ]:
os.chdir("/content")

In [ ]:
!git clone https://github.com/shirtsgroup/analyze_foldamers.git

In [ ]:
os.chdir("analyze_foldamers")

In [ ]:
!python setup.py install

In [ ]:
os.chdir("/content")

In [ ]:
!conda install mdtraj mpi4py numpy openmm openmmtools physical_validation -y

In [ ]:
os.makedirs('Project')
os.chdir('Project')

In [ ]:
!pip install  -U git+https://github.com/shirtsgroup/cg_openmm

In [ ]:
import numpy
numpy.__file__

In [ ]:
!ls /usr/local/lib/python3.10/dist-packages/cg_openmm*

In [ ]:
!rm -rf /usr/local/lib/python3.10/dist-packages/cg_openmm*

In [ ]:
from simtk import unit
from cg_openmm.cg_model.cgmodel import CGModel

# Specify backbone (bb) and sidechain (sc) particle parameters:
sigma = 0.3 * unit.nanometer
epsilon = 2 * unit.kilojoule_per_mole
mass = 100 * unit.amu

bb = {"particle_type_name": "bb", "sigma": sigma, "epsilon": epsilon, "mass": mass}
sc = {"particle_type_name": "sc", "sigma": sigma, "epsilon": epsilon, "mass": mass}

# Define monomer (residue):
A = {
    "monomer_name": "A",
    "particle_sequence": [bb, sc],
    "bond_list": [[0, 1]],
    "start": 0,
    "end": 0}

# Specify bonded parameters:
bond_lengths = {
    "default_bond_length": 0.35 * unit.nanometer,
    "bb_bb_bb_bond_length": 0.40 * unit.nanometer}

bond_force_constants = {
    "default_bond_force_constant": 1000 * unit.kilojoule_per_mole / unit.nanometer**2}

equil_bond_angles = {
    "default_equil_bond_angle": 120.0 * unit.degrees,
    "bb_bb_bb_equil_bond_angle": 150.0 * unit.degrees}

bond_angle_force_constants = {
    "default_bond_angle_force_constant": 100.0 * unit.kilojoule_per_mole / unit.radian**2}

torsion_phase_angles = {
    "default_torsion_phase_angle": 150 * unit.degrees}

torsion_force_constants = {
    "default_torsion_force_constant": 2.0 * unit.kilojoule_per_mole,
    "bb_bb_bb_bb_torsion_force_constant": 5.0 * unit.kilojoule_per_mole}

torsion_periodicities = {
    "default_torsion_periodicity": 1}

# Define oligomer sequence:
sequence = 12 * [A]

# Initial particle positions determined from random builder

cgmodel = CGModel(
    particle_type_list=[bb, sc],
    bond_lengths=bond_lengths,
    bond_force_constants=bond_force_constants,
    bond_angle_force_constants=bond_angle_force_constants,
    torsion_force_constants=torsion_force_constants,
    equil_bond_angles=equil_bond_angles,
    torsion_phase_angles=torsion_phase_angles,
    torsion_periodicities=torsion_periodicities,
    include_nonbonded_forces=True,
    include_bond_forces=True,
    include_bond_angle_forces=True,
    include_torsion_forces=True,
    constrain_bonds=False,
    sequence=sequence,
    monomer_types=[A],
)